# Week 30, Day 6: Forge: Week 30 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w030/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
print("1. sodium", 0 - 50, "mV (inward), potassium", 0 - (-77), "mV (outward)")
print("2. alpha =", 0.8 / 2, "per ms, beta =", round(0.2 / 2, 3), "per ms")
print("3.", 3 ** 3, "times faster")

### Exercise: Part A: kinetics from clamp records

Complete the loop: for each record g, compute n∞ from the last 40 samples, find the τ on the grid `taus` that minimises the squared error, and append both. Then compute `alpha_fit` and `beta_fit`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
gK = 36.0
def an_(V): V = V + 1e-7; return 0.01 * (V + 55) / (1 - np.exp(-(V + 55) / 10))
def bn_(V): return 0.125 * np.exp(-(V + 65) / 80)

r = np.random.default_rng(1952)
t = np.arange(0, 25, 0.05)
Vs = np.arange(-80, 51, 10.0)
n0 = an_(-65.0) / (an_(-65.0) + bn_(-65.0))
data = np.array([gK * (an_(V) / (an_(V) + bn_(V)) - (an_(V) / (an_(V) + bn_(V)) - n0) * np.exp(-t * (an_(V) + bn_(V)))) ** 4
                 + r.normal(0, 0.1, t.size) for V in Vs])

taus = np.linspace(0.2, 12, 1181)
ninf_list, tau_list = [], []
for g in data:
    ni = ...
    errs = ...
    ninf_list.append(ni); tau_list.append(taus[int(np.argmin(errs))])
ninf_fit, tau_fit = np.array(ninf_list), np.array(tau_list)
alpha_fit = ...
beta_fit = ...
for V, a, b in zip(Vs, alpha_fit, beta_fit):
    print(f"{V:5.0f} mV: alpha {a:.4f} (true {an_(V):.4f}), beta {b:.4f} (true {bn_(V):.4f})")

In [ ]:
import numpy as np
def test_ninf():
    "ninf_fit uses the mean of the last 40 samples of each record"
    assert np.allclose(ninf_fit, np.maximum(data[:, -40:].mean(1), 1e-6) ** 0.25 / gK ** 0.25)

def test_rates():
    "alpha_fit and beta_fit come from the fits and match the true rates within 10% from -50 mV upwards"
    assert np.allclose(alpha_fit, ninf_fit / tau_fit) and np.allclose(beta_fit, (1 - ninf_fit) / tau_fit)
    sel = Vs >= -50
    assert np.all(np.abs(alpha_fit[sel] / an_(Vs[sel]) - 1) < 0.1) and np.all(np.abs(beta_fit[sel] / bn_(Vs[sel]) - 1) < 0.1)

test_ninf()
test_rates()
print("All checks passed")

### Exercise: Part B: from clamp to spikes

`simulate(use_fit)` runs the model with either the true or the fitted potassium rates. Find the spike times (upward crossings of 0 mV) of both runs, `spikes_true` and `spikes_fit`, their peak voltages, and mean interspike intervals `isi_true` and `isi_fit`.

In [ ]:
import math

def other_rates(V):
    V = V + 1e-7
    return (0.1 * (V + 40) / (1 - math.exp(-(V + 40) / 10)), 4.0 * math.exp(-(V + 65) / 18),
            0.07 * math.exp(-(V + 65) / 20), 1 / (1 + math.exp(-(V + 35) / 10)))

def simulate(use_fit, I=10.0, T=60.0, dt=0.01):
    V, n, m, h = -65.0, n0, 0.0529, 0.5961
    out = np.empty(int(T / dt))
    for k in range(len(out)):
        am, bm, ah, bh = other_rates(V)
        if use_fit:
            an, bn = np.interp(V, Vs, alpha_fit), np.interp(V, Vs, beta_fit)
        else:
            an, bn = an_(V), bn_(V)
        dV = (I if k * dt >= 5 else 0.0) - 120 * m**3 * h * (V - 50) - gK * n**4 * (V + 77) - 0.3 * (V + 54.387)
        n += dt * (an * (1 - n) - bn * n); m += dt * (am * (1 - m) - bm * m); h += dt * (ah * (1 - h) - bh * h)
        V += dt * dV
        out[k] = V
    return out

def spike_times(v, dt=0.01):
    return ...

v_true, v_fit = simulate(False), simulate(True)
spikes_true, spikes_fit = spike_times(v_true), spike_times(v_fit)
peak_true, peak_fit = v_true.max(), v_fit.max()
isi_true, isi_fit = np.diff(spikes_true).mean(), np.diff(spikes_fit).mean()
print(f"original: {len(spikes_true)} spikes, peak {peak_true:.1f} mV, mean interval {isi_true:.2f} ms")
print(f"fitted:   {len(spikes_fit)} spikes, peak {peak_fit:.1f} mV, mean interval {isi_fit:.2f} ms")
tt = np.arange(len(v_true)) * 0.01
plt.plot(tt, v_true, label="original"); plt.plot(tt, v_fit, "--", label="fitted potassium kinetics")
plt.xlabel("ms"); plt.ylabel("V (mV)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_spikes():
    "both models fire the same number of spikes in 60 ms, with peaks within 2 mV"
    assert len(spikes_true) == len(spikes_fit) >= 3 and abs(peak_true - peak_fit) < 2

def test_isi():
    "the fitted model's mean interspike interval is within 10% of the original's"
    assert abs(isi_fit - isi_true) / isi_true < 0.1

test_spikes()
test_isi()
print("All checks passed")